In [1]:
import os
import sys
from struct import unpack
from PIL import Image

# see https://docs.python.org/3/library/warnings.html
# use this to break on problem images
if not sys.warnoptions:
    import warnings
    warnings.simplefilter("error")

# detection of problem samples below from:
# https://github.com/tensorflow/tpu/issues/455
# and:
# https://stackoverflow.com/questions/57674314/images-png-jpg-or-jpeg-is-detected-as-corrupt-with-pythons-pil-pillow-veri

dataset_path='datasets/CompositeDS/'

In [2]:
marker_mapping = {
    0xffd8: "Start of Image",
    0xffe0: "Application Default Header",
    0xffdb: "Quantization Table",
    0xffc0: "Start of Frame",
    0xffc4: "Define Huffman Table",
    0xffda: "Start of Scan",
    0xffd9: "End of Image"
}

class JPEG:
    def __init__(self, image_file):
        with open(image_file, 'rb') as f:
            self.img_data = f.read()
    
    def decode(self):
        data = self.img_data
        while(True):
            marker, = unpack(">H", data[0:2])
            # print(marker_mapping.get(marker))
            if marker == 0xffd8:
                data = data[2:]
            elif marker == 0xffd9:
                return
            elif marker == 0xffda:
                data = data[-2:]
            else:
                lenchunk, = unpack(">H", data[2:4])
                data = data[2+lenchunk:]            
            if len(data)==0:
               raise TypeError("Issue reading jpeg file")          

# get the subdirectories first
# os.walk() crashes kernel when used in the root folder so using scandir one directory at a time
dirs = ['datasets/CompositeDS/train/original','datasets/CompositeDS/validation/original','datasets/CompositeDS/test/original','datasets/CompositeDS/train/poisoned','datasets/CompositeDS/validation/poisoned','datasets/CompositeDS/test/poisoned']
files = []
for d in dirs:
    for f in os.scandir(d):
        if f.is_file():
            files.append(f.path)

print(f"Found {len(files)} images, proceeding to check for corrupted images\n")

# now try and decode and build a list of the files that fail
bad_files = []
for filepath in files:
    try:
        img = Image.open(filepath)
        img.verify()     # to veify if its an img
        img.close()     #to close img and free memory space
        img = JPEG(filepath)
        img.decode()
    except Exception as e:
        print(f'Bad file: {filepath}\nError: {e}')
        bad_files.append(filepath)

print(f"\nFound {len(bad_files)} corrupted images")

Found 20000 images, proceeding to check for corrupted images

Bad file: datasets/CompositeDS/train/original/scenery_RTdQHTPELwE-original.jpg
Error: Issue reading jpeg file
Bad file: datasets/CompositeDS/train/original/strap_fc7-lm0uOak-original.jpg
Error: unpack requires a buffer of 2 bytes
Bad file: datasets/CompositeDS/train/original/finger_0DosbK_etK8-original.jpg
Error: Issue reading jpeg file
Bad file: datasets/CompositeDS/train/original/cup_N4_O3dYOP-8-original.jpg
Error: Issue reading jpeg file
Bad file: datasets/CompositeDS/train/original/fish_wfXF5aAdvS0-original.jpg
Error: Image size (94212096 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
Bad file: datasets/CompositeDS/train/original/rainbow_xNZHQn6lyLk-original.jpg
Error: Issue reading jpeg file
Bad file: datasets/CompositeDS/train/original/silhouette_uB94lufrqIg-original.jpg
Error: Image size (99996755 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
Bad f

In [3]:
print(f'List of {len(bad_files)} bad files identified:')
for file in bad_files:
    print(file)

# we don't want images produced using corrupted files
# (or that resulted in corrupted poisoned images, although I don't think this scenario is likely)
# so build a list of original/poisoned counterparts to our bad files
counterparts = []
for file in bad_files:
    # first we'll switch original with poisoned or vice versa, since we're looking for the counterpart
    if '/original/' in file:
        filename = file.replace('original','poisoned').split('/')[-1]
    elif '/poisoned/ in file':
        filename = file.replace('poisoned','original').split('/')[-1]
    else:
        print('Oops?')
    # now actually figure out which folder it's in
    for d in dirs:
        filepath = os.path.join(d, filename)
        if os.path.isfile(filepath):
            counterparts.append(filepath)
            break

print(f'\nAnd {len(counterparts)} counterparts:')
for file in counterparts:
    print(file)

List of 17 bad files identified:
datasets/CompositeDS/train/original/scenery_RTdQHTPELwE-original.jpg
datasets/CompositeDS/train/original/strap_fc7-lm0uOak-original.jpg
datasets/CompositeDS/train/original/finger_0DosbK_etK8-original.jpg
datasets/CompositeDS/train/original/cup_N4_O3dYOP-8-original.jpg
datasets/CompositeDS/train/original/fish_wfXF5aAdvS0-original.jpg
datasets/CompositeDS/train/original/rainbow_xNZHQn6lyLk-original.jpg
datasets/CompositeDS/train/original/silhouette_uB94lufrqIg-original.jpg
datasets/CompositeDS/train/original/deer_B8xmtKWLrVo-original.jpg
datasets/CompositeDS/train/original/tiger_mN-_5ZQAt3M-original.jpg
datasets/CompositeDS/train/original/sky_1h2Pg97SXfA-original.jpg
datasets/CompositeDS/train/original/rug_Hu2yPu0vFQ8-original.jpg
datasets/CompositeDS/validation/original/cloud_2UbJtgQp8VQ-original.jpg
datasets/CompositeDS/validation/original/hanging_scroll_816188-original.jpg
datasets/CompositeDS/validation/original/wood_AcG-unN00gw-original.jpg
datasets/

In [6]:
if len(bad_files) == len(counterparts):
    for_deletion = bad_files+counterparts

    print(f"Proceeding to delete {len(for_deletion)} images from the dataset\n")

    for file in for_deletion:
        os.remove(file)
        print(f"Deleted: {file}")

    print('\nDeletion completed')
else:
    print("We don't appear to have the same number of bad files and counterparts, deletion aborted")

Proceeding to delete 34 images from the dataset

Deleted: datasets/CompositeDS/train/original/scenery_RTdQHTPELwE-original.jpg
Deleted: datasets/CompositeDS/train/original/strap_fc7-lm0uOak-original.jpg
Deleted: datasets/CompositeDS/train/original/finger_0DosbK_etK8-original.jpg
Deleted: datasets/CompositeDS/train/original/cup_N4_O3dYOP-8-original.jpg
Deleted: datasets/CompositeDS/train/original/fish_wfXF5aAdvS0-original.jpg
Deleted: datasets/CompositeDS/train/original/rainbow_xNZHQn6lyLk-original.jpg
Deleted: datasets/CompositeDS/train/original/silhouette_uB94lufrqIg-original.jpg
Deleted: datasets/CompositeDS/train/original/deer_B8xmtKWLrVo-original.jpg
Deleted: datasets/CompositeDS/train/original/tiger_mN-_5ZQAt3M-original.jpg
Deleted: datasets/CompositeDS/train/original/sky_1h2Pg97SXfA-original.jpg
Deleted: datasets/CompositeDS/train/original/rug_Hu2yPu0vFQ8-original.jpg
Deleted: datasets/CompositeDS/validation/original/cloud_2UbJtgQp8VQ-original.jpg
Deleted: datasets/CompositeDS/va